# Steam Game Genre Analysis
## Fase 1 + 2 + 3 — Volledige Pipeline

**Onderwerp:** *Watter genre werk op Steam?* — watter genre is die gewildste, watter kenmerke dryf sukses, en hoe lyk die kommentaar binne elke genre?

**Datastel:** 33 speletjies oor 15 genres, ~161 000 skoon Engelse resensies (Okt 2021 – Aug 2026)

**Databronne:** Steam API — resensies, speletjiebesonderhede, huidige spelertellings

**Tegnologieë:** Python, pandas, numpy, requests, scipy, matplotlib

**Fase-omvang:** Fase 1 (insameling) + Fase 2 (verwerking) + Fase 3 (EDA, suiwer statistiek — geen VADER/TF-IDF/netwerke nie).

**Herhaalbaarheid:** Verwyder `data/raw/*.csv` en `data/processed/*.csv` en hardloop die notebook weer.

---
## Navorsingsvraag & Hipotese

**Doelwit:** Speletjie-ontwikkelaars en beleggers help om 'n ingelige besluit te neem oor watter genre om in te belê, gebaseer op werklike Steam-data oor gewildheid, sentiment en suksesfaktore per genre.

**Vraag:** *Watter genre werk op Steam?*

**Hipotese:** *Genre bepaal die basislyn-verwagtinge van spelers; suksesfaktore verskil tussen genres.* 'n Ontwikkelaar moet hul mededingers binne hul teikengenre meet, nie die hele speletjiemark nie.

**Sub-vrae:**
1. Watter kenmerke voorspel 'n positiewe resensie? (speeltyd, resensielengte, stemme, aankoop-type)
2. Verskil suksesfaktore per genre?
3. Hoe ontwikkel spelertevredenheid oor tyd? Kan 'n 'verlossingsboog'  gemeet word?
4. Wat is die tipiese seisoenspatrone?

**Fases in hierdie notebook:**
- **Fase 1:** Aktiewe data-insameling via Steam API (33 speletjies × 50 bladsye Engels).
- **Fase 2:** Skoonmaak, kenmerk-ingenieurswese, genre one-hot.
- **Fase 3:** Verkennende data-analise — basiese statistieke, formele toetse, patrone, tendense, uitdagings.

---
## Etiese Oorwegings & Bronnelys

**Eties:** Slegs openbare data via Steam se amptelike API. Geen privaatgebruikersinligting word versamel nie. Geen poging om gebruikersgedrag te manipuleer nie.

**Bronnelys:**
1. **Steam API** — https://steamcommunity.com/dev
2. **Steam Web API (appreviews)** — https://partner.steamgames.com/doc/store/getreviews
3. **pandas** — McKinney, W. (2010). Data Structures for Statistical Computing in Python. *9th Python in Science Conference*.
4. **requests** — https://requests.readthedocs.io
5. **scipy** — Virtanen et al. (2020). SciPy 1.0: fundamental algorithms for scientific computing in Python. *Nature Methods* 17, 261–272.
6. **VADER-Sentiment** — Hutto, C.J. & Gilbert, E.E. (2014). VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text. *ICWSM-14*.

In [ ]:
%matplotlib inline
import os, sys, time, re
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# --- Pade ---
PROJECT_DIR = os.path.abspath('.')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')

def ensure_dirs():
    for d in [RAW_DIR, PROCESSED_DIR]:
        os.makedirs(d, exist_ok=True)

def safe_request(url, params=None, max_retries=3, delay=2):
    """HTTP GET met lineêre back-off. Gee None by faling."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(delay * (attempt + 1))
        except requests.RequestException:
            time.sleep(delay * (attempt + 1))
    return None

ensure_dirs()
print(f'Opstelling gereed. Pade: {PROJECT_DIR}')

In [ ]:
# --- 33 speletjies ---
GAMES = {
    1245620: 'Elden Ring',
    1086940: "Baldur's Gate 3",
    730: 'Counter-Strike 2',
    2357570: 'Overwatch 2',
    2767030: 'Marvel Rivals',
    1091500: 'Cyberpunk 2077',
    275850: "No Man's Sky",
    578080: 'PUBG: BATTLEGROUNDS',
    1938090: 'Call of Duty HQ',
    553850: 'Helldivers 2',
    292030: 'The Witcher 3',
    489830: 'Skyrim SE',
    2054970: "Dragon's Dogma 2",
    377160: 'Fallout 4',
    1172470: 'Apex Legends',
    1085660: 'Destiny 2',
    359550: 'Rainbow Six Siege',
    1517290: 'Battlefield 2042',
    2807960: 'Battlefield 6',
    440: 'Team Fortress 2',
    2215430: 'Ghost of Tsushima',
    1593500: 'God of War',
    1174180: 'Red Dead Redemption 2',
    601150: 'Devil May Cry 5',
    281990: 'Stellaris',
    289070: "Sid Meier's Civilization VI",
    1142710: 'Total War: WARHAMMER III',
    1466860: 'Age of Empires IV',
    252490: 'Rust',
    346110: 'ARK: Survival Evolved',
    892970: 'Valheim',
    242760: 'The Forest',
    1623730: 'Palworld',
}
GAME_IDS = list(GAMES.keys())

# --- 15 genres ---
GENRES = {
    'RPG':            [1086940, 1091500, 1245620, 292030, 489830, 2054970, 377160, 892970],
    'Shooter':        [730, 578080, 553850, 1938090, 2357570, 1172470, 1085660, 359550, 1517290, 2807960, 440],
    'Hero_Shooter':   [2357570, 2767030, 440],
    'Battle_Royale':  [578080, 1938090, 1172470],
    'Action':         [1245620, 1091500, 275850, 553850, 2215430, 1593500, 1174180, 601150, 292030, 489830, 2054970, 377160, 440, 252490, 346110, 892970, 242760],
    'Strategy':       [1086940, 281990, 289070, 1142710, 1466860, 359550],
    'Adventure':      [1086940, 275850, 292030, 489830, 377160, 2215430, 1593500, 1174180, 346110, 892970, 242760],
    'Survival':       [275850, 252490, 346110, 892970, 242760, 1623730],
    'Free_to_Play':   [730, 2357570, 2767030, 578080, 1172470, 440],
    'Third_Person':   [1245620, 1086940, 292030, 2215430, 1593500, 1174180, 601150, 2054970, 553850, 2767030, 1623730, 489830, 377160, 275850, 892970, 346110, 578080, 1142710],
    'First_Person':   [730, 2357570, 440, 1517290, 2807960, 1938090, 359550, 1085660, 1172470, 242760, 489830, 377160, 275850, 892970, 346110, 578080],
    'Top_Down':       [1086940, 281990, 289070, 1142710, 1466860],
    'Single_Player':  [1245620, 1086940, 1091500, 292030, 489830, 377160, 2215430, 1593500, 1174180, 601150, 2054970, 275850, 892970, 242760, 2807960, 1938090, 289070, 1466860],
    'Multiplayer':    [730, 2357570, 2767030, 440, 578080, 1938090, 359550, 1085660, 1172470, 1517290, 2807960, 553850, 252490, 346110, 892970, 289070, 1466860, 1142710],
    'Indie':          [252490, 346110, 892970, 242760, 1623730],
}
GENRE_IDS = list(GENRES.keys())

# --- Omgekeerde indeks: app_id -> [genre tags] ---
GAME_GENRES = {}
for genre, ids in GENRES.items():
    for app_id in ids:
        GAME_GENRES.setdefault(app_id, []).append(genre)

print(f'{len(GAMES)} speletjies oor {len(GENRES)} genres.')
for g, ids in GENRES.items():
    print(f'  {g:15s} ({len(ids):>2}): {", ".join(GAMES[a] for a in ids)}')

---
# Fase 1: Aktiewe Data-Insameling (Steam API)

Ons skraap self — geen klaargemaakte CSV word afgelaai nie.

### Skraping-parameters

| Parameter | Waarde | Regverdiging |
|---|---|---|
| **Taal** | `english` | VADER en TF-IDF (Fase 4) werk slegs op Engels |
| **Bladsye** | 50 (5 000 resensies) | Nuutste 5 000 dek mediaan ~1 jaar per speletjie |
| **Aankoop** | `purchase_type=all` | Alle resensies, nie net betaalde nie |
| **Tydreeks** | `day_range=9999` | Alle datums, paginering beperk natuurlikerwys tot nuutste |
| **Tempo** | 0.3 s | Vriendelik teenoor API |
| **Deduplikasie** | `seen_ids` per spel | Voorkom duplikate |
| **Stop-reël** | 3 leë bladsye | Einde van resensies bereik |

### Brondata

- `app_details.csv` — 33 rye, een per speletjie (naam, prys, genres, aanbevelings)
- `reviews.csv` — ~163 000 rye, 18 kolomme (resensie-metadata)
- `player_counts.csv` — 33 rye, huidige aktiewe spelers

In [ ]:
STEAM_APP_DETAILS_URL = 'https://store.steampowered.com/api/appdetails'
STEAM_REVIEWS_URL = 'https://store.steampowered.com/appreviews'
STEAM_PLAYER_COUNT_URL = 'https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/'

def fetch_app_details(app_id):
    data = safe_request(STEAM_APP_DETAILS_URL, {'appids': app_id})
    if data and str(app_id) in data:
        d = data[str(app_id)].get('data', {})
        return {
            'app_id': app_id,
            'name': d.get('name', ''),
            'release_date': d.get('release_date', {}).get('date', '') if d.get('release_date') else '',
            'developers': ', '.join(d.get('developers', [])),
            'publishers': ', '.join(d.get('publishers', [])),
            'genres': ', '.join(g['description'] for g in d.get('genres', [])),
            'categories': ', '.join(c['description'] for c in d.get('categories', [])),
            'price': d.get('price_overview', {}).get('final', 0) if d.get('price_overview') else 0,
            'metacritic_score': d.get('metacritic', {}).get('score', None) if d.get('metacritic') else None,
            'recommendations': d.get('recommendations', {}).get('total', 0) if d.get('recommendations') else 0,
        }
    return None

def scrape_reviews(app_id, max_pages=50, reviews_per_page=100):
    all_reviews = []
    cursor = '*'
    seen_ids = set()
    empty_page_count = 0

    for page in range(max_pages):
        params = {
            'json': 1,
            'language': 'english',
            'num_per_page': reviews_per_page,
            'purchase_type': 'all',
            'day_range': 9999,
            'cursor': cursor,
        }
        data = safe_request(f'{STEAM_REVIEWS_URL}/{app_id}', params)
        if not data or not data.get('success'):
            break

        reviews = data.get('reviews', [])
        if not reviews:
            break

        new_count = 0
        for r in reviews:
            rid = r.get('recommendationid', '')
            if rid in seen_ids:
                continue
            seen_ids.add(rid)
            new_count += 1
            all_reviews.append({
                'app_id': app_id,
                'game_name': GAMES.get(app_id, ''),
                'review_id': rid,
                'author_id': r.get('author', {}).get('steamid', ''),
                'num_games_owned': r.get('author', {}).get('num_games_owned', 0),
                'num_reviews': r.get('author', {}).get('num_reviews', 0),
                'playtime_forever': r.get('author', {}).get('playtime_forever', 0),
                'language': r.get('language', ''),
                'review_text': r.get('review', ''),
                'timestamp_created': r.get('timestamp_created', 0),
                'voted_up': r.get('voted_up', False),
                'votes_up': r.get('votes_up', 0),
                'votes_funny': r.get('votes_funny', 0),
                'weighted_vote_score': r.get('weighted_vote_score', ''),
                'steam_purchase': r.get('steam_purchase', False),
                'received_for_free': r.get('received_for_free', False),
                'written_during_early_access': r.get('written_during_early_access', False),
            })

        if new_count == 0:
            empty_page_count += 1
            if empty_page_count >= 3:
                break
        else:
            empty_page_count = 0

        cursor = data.get('cursor', '*')
        time.sleep(0.3)

    return all_reviews

def fetch_player_count(app_id):
    data = safe_request(STEAM_PLAYER_COUNT_URL, {'appid': app_id, 'format': 'json'})
    if data and 'response' in data:
        return {
            'app_id': app_id,
            'game_name': GAMES.get(app_id, ''),
            'player_count': data['response'].get('player_count', 0),
            'game_id': data['response'].get('game_id', app_id),
        }
    return None

def scrape_all():
    """Skraap 33 speletjies: app details, resensies (50 bladsye elk), speler-tellings."""
    ensure_dirs()

    # 1) App details
    app_details = []
    for app_id in GAME_IDS:
        details = fetch_app_details(app_id)
        if details:
            app_details.append(details)
            print(f"  Details: {details['name']}")
        time.sleep(0.3)
    df_details = pd.DataFrame(app_details)
    df_details.to_csv(f'{RAW_DIR}/app_details.csv', index=False)
    print(f"\nStoor app details vir {len(app_details)} speletjies")

    # 2) Resensies
    all_reviews = []
    for app_id in GAME_IDS:
        name = GAMES[app_id]
        print(f"\nSkraap resensies vir {name} (app_id={app_id})...")
        game_reviews = scrape_reviews(app_id)
        all_reviews.extend(game_reviews)
        print(f"  {len(game_reviews)} resensies")
    df_reviews = pd.DataFrame(all_reviews)
    df_reviews.to_csv(f'{RAW_DIR}/reviews.csv', index=False)
    print(f"\nStoor {len(df_reviews)} totale resensies")

    # 3) Speler-tellings
    player_snapshots = []
    for app_id in GAME_IDS:
        pc = fetch_player_count(app_id)
        if pc:
            player_snapshots.append(pc)
            print(f"  {GAMES[app_id]}: {pc['player_count']:,} spelers")
        time.sleep(0.3)
    df_players = pd.DataFrame(player_snapshots)
    df_players.to_csv(f'{RAW_DIR}/player_counts.csv', index=False)
    print(f"\nStoor speler-tellings vir {len(player_snapshots)} speletjies")

    return df_details, df_reviews, df_players

print('Skraper-funksies gereed.')

In [ ]:
# Voer die skraping uit (~15-25 minute vir 33 speletjies × 50 bladsye)
df_details, df_reviews, df_players = scrape_all()
print(f"\nRou data: {len(df_details)} besonderhede, {len(df_reviews)} resensies, {len(df_players)} speler-tellings")

In [ ]:
print('=== App details (voorskou) ===')
print(df_details[['app_id', 'name', 'release_date', 'price', 'recommendations']].to_string(index=False))

print('\n=== Resensies (kolomme + datatipes) ===')
print(f'Vorm: {df_reviews.shape}')
print(df_reviews.dtypes)

print('\n=== Taal-verspreiding (Engels moet dominant wees) ===')
print(df_reviews['language'].value_counts().head(10))

print('\n=== Resensies per speletjie ===')
print(df_reviews.groupby('game_name').size().sort_values(ascending=False).to_string())

print('\n=== Speler-tellings (top 10) ===')
print(df_players.sort_values('player_count', ascending=False).head(10).to_string(index=False))

---
# Fase 2: Data Skoonmaak & Verwerking

### Skoonmaakstappe

| Stap | Wat | Hoekom |
|---|---|---|
| **Deduplikasie** | `drop_duplicates(subset='review_id')` | Steam-paginering kan duplikate oplewer |
| **Lengte-filter** | Verwyder resensies < 10 karakters | Te kort om mening te dra |
| **Karakter-ratio** | Behou rye waar ≥ 50% letters/spasies is | Verwyder nie-tekstuele inhoud |
| **Tipe-koersie** | Numeriese → `float`, boole → `bool` | Konsistente tipes vir analise |
| **Datum-ontleding** | `timestamp_created` → `review_date` | Maak tyd-tendense moontlik |
| **Speeltyd-uitskieters** | Knip by 99ste persentiel | Langstert kan gemiddeldes verdraai |
| **Teksskoonmaak** | Verwyder URL's, nie-alfakarakters; spasie-genormaliseer | Skoner teks vir NLP |
| **Kenmerk-ingenieurswese** | `review_length`, `word_count` | Maklik verwerkbare maatstawwe |
| **Genre one-hot** | 15 kolomme `genre_{GENRE}` via `GAME_GENRES` | Laat genre-gebaseerde groepering toe |

**Verwagte resultaat:** ~161 000 skoon resensies, ~36 kolomme (15 genre + 21 resensie-kenmerke).

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s\'.!?,;-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_reviews(df):
    if df is None or df.empty:
        return df

    df = df.copy()
    df = df.drop_duplicates(subset='review_id')

    # Verwyder leeg / te kort / nie-tekstueel
    df = df[df['review_text'].notna() & (df['review_text'].str.strip() != '')]
    df = df[df['review_text'].str.len() >= 10]
    char_threshold = 0.5
    df = df[df['review_text'].apply(
        lambda x: sum(c.isalpha() or c.isspace() for c in str(x)) / max(len(str(x)), 1) > char_threshold
    )]

    # Datum-ontleding
    df['timestamp_created'] = pd.to_numeric(df['timestamp_created'], errors='coerce')
    df = df[df['timestamp_created'].notna()]
    df['review_date'] = pd.to_datetime(df['timestamp_created'], unit='s')
    df['review_year'] = df['review_date'].dt.year
    df['review_month'] = df['review_date'].dt.month
    df['review_day_of_week'] = df['review_date'].dt.dayofweek

    # Speeltyd-uitskieters by 99ste persentiel afkap
    df['playtime_forever'] = pd.to_numeric(df['playtime_forever'], errors='coerce').fillna(0).astype(float)
    p99 = df['playtime_forever'].quantile(0.99)
    df.loc[df['playtime_forever'] > p99, 'playtime_forever'] = p99

    # Tipe-koersie
    df['voted_up'] = df['voted_up'].astype(bool)
    df['steam_purchase'] = df['steam_purchase'].astype(bool)
    df['written_during_early_access'] = df['written_during_early_access'].astype(bool)

    # Skoon teks + lengte-kenmerke
    df['review_text_clean'] = df['review_text'].apply(clean_text)
    df['review_length'] = df['review_text'].str.len()
    df['word_count'] = df['review_text'].str.split().str.len()

    # language-kolom is oorbodig (alles is Engels)
    if 'language' in df.columns:
        df = df.drop(columns=['language'])

    # 15 genre one-hot kolomme
    genres_for_app = df['app_id'].map(GAME_GENRES)
    for genre in GENRE_IDS:
        df[f'genre_{genre}'] = genres_for_app.apply(
            lambda tags: 1 if isinstance(tags, list) and genre in tags else 0
        )

    return df

print('Skoonmaak-funksies gereed.')

In [ ]:
# Pas skoonmaak toe
raw_count = len(df_reviews)
print(f'Rou resensies: {raw_count:,}')

df_clean = clean_reviews(df_reviews)
print(f'Na skoonmaak: {len(df_clean):,}')
print(f'Verwyder deur skoonmaak: {raw_count - len(df_clean):,} ({100*(raw_count-len(df_clean))/raw_count:.1f}%)')

# Stoor
out_path = f'{PROCESSED_DIR}/reviews_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'\nGestoor: {out_path}')

In [ ]:
print('=== Vorm ===')
print(f'Res: {df_clean.shape[0]:,} rye, {df_clean.shape[1]} kolomme')

print('\n=== Kolomme en tipes ===')
info = pd.DataFrame({
    'kolom': df_clean.columns,
    'dtype': [str(df_clean[c].dtype) for c in df_clean.columns],
    'nie-null': [df_clean[c].notna().sum() for c in df_clean.columns],
    'uniek': [df_clean[c].nunique() for c in df_clean.columns],
})
print(info.to_string(index=False))

print('\n=== Datakwaliteit-opsomming ===')
print(f'  Speletjies: {df_clean["app_id"].nunique()}')
print(f'  Resensies: {len(df_clean):,}')
print(f'  Positief: {df_clean["voted_up"].sum():,} ({100*df_clean["voted_up"].mean():.1f}%)')
print(f'  Gem. speeltyd: {df_clean["playtime_forever"].mean():.0f} minute = {df_clean["playtime_forever"].mean()/60:.0f} uur')
print(f'  Mediaan speeltyd: {df_clean["playtime_forever"].median():.0f} minute = {df_clean["playtime_forever"].median()/60:.0f} uur')
print(f'  Gem. woorde: {df_clean["word_count"].mean():.1f}')
print(f'  Gem. resensielengte: {df_clean["review_length"].mean():.0f} karakters')
print(f'  Datumreeks: {df_clean["review_date"].min().date()} tot {df_clean["review_date"].max().date()}')
print(f'  Steam-aankope: {100*df_clean["steam_purchase"].mean():.1f}%')
print(f'  Vroeg-toegang: {100*df_clean["written_during_early_access"].mean():.1f}%')

genre_cols = [c for c in df_clean.columns if c.startswith("genre_")]
print(f'\n  Genre one-hot kolomme: {len(genre_cols)}')
for c in genre_cols:
    n = df_clean[c].sum()
    print(f'    {c:20s}: {n:>7,} rye ({100*n/len(df_clean):.1f}%)')

**Status Fase 2:** GEREED. ~161 000 skoon Engelse resensies, 36 kolomme, 15 genre one-hot, datums Okt 2021 – Aug 2026.

**Brug na Fase 3:** Ons het nou 'n skoon datastel. Volgende: leer dit ken deur basiese statistieke, formele toetse, en patroon-ontleding.

---
# Fase 3: Verkennende Data Analise (EDA)

Fase 3 het drie sub-doelwitte:

1. **§ 3.1 Basiese Statistieke** — totale oorsig, per speletjie, per genre.
2. **§ 3.2 Formele Statistiese Toetse** — t-toetse, ANOVA, chi-kwadraat, korrelasie.
3. **§ 3.3 Patrone, Tendense & Uitdagings** — verdelings, tyd-tendense, uitskieters, resensiebomme.

---
## § 3.1 Basiese Statistieke

In [ ]:
def basic_statistics(df):
    return {
        'total_reviews': len(df),
        'positive_reviews': int(df['voted_up'].sum()),
        'negative_reviews': int((~df['voted_up']).sum()),
        'positive_pct': round(df['voted_up'].mean() * 100, 2),
        'avg_playtime_min': round(df['playtime_forever'].mean(), 1),
        'median_playtime_min': round(df['playtime_forever'].median(), 1),
        'avg_review_length': round(df['review_length'].mean(), 1),
        'avg_word_count': round(df['word_count'].mean(), 1),
        'date_range': f"{df['review_date'].min().date()} tot {df['review_date'].max().date()}",
        'num_games': df['app_id'].nunique(),
        'steam_purchase_pct': round(df['steam_purchase'].mean() * 100, 1),
    }

stats_summary = basic_statistics(df_clean)
for k, v in stats_summary.items():
    print(f'  {k:20s}: {v}')

In [ ]:
per_game_rows = []
for app_id, name in GAMES.items():
    g = df_clean[df_clean['app_id'] == app_id]
    if g.empty:
        continue
    per_game_rows.append({
        'game': name,
        'total_reviews': len(g),
        'positive': int(g['voted_up'].sum()),
        'negative': int((~g['voted_up']).sum()),
        'positive_pct': round(g['voted_up'].mean() * 100, 1),
        'avg_playtime_h': round(g['playtime_forever'].mean() / 60, 0),
        'avg_word_count': round(g['word_count'].mean(), 1),
        'steam_purchase_pct': round(g['steam_purchase'].mean() * 100, 1),
    })
per_game_df = pd.DataFrame(per_game_rows).sort_values('positive_pct', ascending=False)
per_game_df = per_game_df.reset_index(drop=True)
per_game_df

In [ ]:
per_genre_rows = []
for genre in GENRE_IDS:
    g = df_clean[df_clean[f'genre_{genre}'] == 1]
    if g.empty:
        continue
    per_genre_rows.append({
        'genre': genre.replace('_', ' '),
        'total_reviews': len(g),
        'positive': int(g['voted_up'].sum()),
        'negative': int((~g['voted_up']).sum()),
        'positive_pct': round(g['voted_up'].mean() * 100, 1),
        'avg_playtime_h': round(g['playtime_forever'].mean() / 60, 0),
        'avg_word_count': round(g['word_count'].mean(), 1),
        'num_games': g['app_id'].nunique(),
    })
per_genre_df = pd.DataFrame(per_genre_rows).sort_values('positive_pct', ascending=False)
per_genre_df = per_genre_df.reset_index(drop=True)
per_genre_df

### Voorlopige bevinding: 4 sentiment-clusters

Die 33 speletjies val natuurlik in 4 groepe volgens positiewe %:

| Groep | Positiewe % | Voorbeelde |
|---|---|---|
| **Loved** | 89 – 99% | Elden Ring, Cyberpunk 2077, BG3, Palworld, Witcher 3 |
| **Good** | 79 – 89% | Ghost of Tsushima, RDR2, Civ VI, AoE IV |
| **Mixed** | 50 – 68% | Destiny 2, Rust, ARK, PUBG, R6 Siege, Apex, CS2 |
| **Disliked** | 10 – 46% | Overwatch 2, TW:WH3, FO4, BF2042, BF6, CoD HQ, Helldivers 2 |

**Insig:** Die grootste determinant van 'n resensie is nie objektiewe kwaliteit nie, maar gemeenskapsverwagtinge en kontroversies. Cyberpunk 2077 (na updates) en Helldivers 2 (tydens bom) sit by die uiterstes.

---
## § 3.2 Formele Statistiese Toetse

Vrae wat ons met formele toetse beantwoord:

| Toets | Vraag |
|---|---|
| Welch se t-toets (woorde) | Is negatiewe resensies langer as positiewe? |
| Welch se t-toets (speeltyd) | Speel langterspelers strenger? |
| Eenrigting-ANOVA (woorde per genre) | Skryf sommige genres meer detail? |
| Eenrigting-ANOVA (woorde per speletjie) | Verskil resensielengte tussen speletjies? |
| Chi-kwadraat (genre × voted_up) | Hang genre en sentiment saam? |
| Spearman-korrelasie | Hoe sterk is speeltyd ↔ sentiment? |

**Betekenis-vlak:** α = 0.05 vir alle toetse.

In [ ]:
pos_words = df_clean[df_clean['voted_up']]['word_count']
neg_words = df_clean[~df_clean['voted_up']]['word_count']
t_stat, p_val = stats.ttest_ind(pos_words, neg_words, equal_var=False)

print("=== Welch se t-toets: woorde per resensie volgens voted_up ===")
print(f"  Positief: gem. {pos_words.mean():.1f} woorde (n={len(pos_words):,})")
print(f"  Negatief: gem. {neg_words.mean():.1f} woorde (n={len(neg_words):,})")
print(f"  Verskil:   {neg_words.mean() - pos_words.mean():.1f} woorde ({100*(neg_words.mean()-pos_words.mean())/pos_words.mean():.0f}% langer)")
print(f"  t = {t_stat:.4f}")
print(f"  p = {p_val:.2e}")
print(f"  Beduidend by α=0.05: {p_val < 0.05}")

In [ ]:
pos_pt = df_clean[df_clean['voted_up']]['playtime_forever'] / 60  # minute -> uur
neg_pt = df_clean[~df_clean['voted_up']]['playtime_forever'] / 60
t_stat, p_val = stats.ttest_ind(pos_pt, neg_pt, equal_var=False)

print("=== Welch se t-toets: speeltyd volgens voted_up ===")
print(f"  Positief: gem. {pos_pt.mean():.0f} uur (n={len(pos_pt):,})")
print(f"  Negatief: gem. {neg_pt.mean():.0f} uur (n={len(neg_pt):,})")
print(f"  Verskil:  {neg_pt.mean() - pos_pt.mean():.0f} uur langer vir negatief")
print(f"  t = {t_stat:.4f}")
print(f"  p = {p_val:.2e}")
print(f"  Beduidend by α=0.05: {p_val < 0.05}")

In [ ]:
groups = [df_clean[df_clean[f'genre_{g}'] == 1]['word_count'].values for g in GENRE_IDS]
f_stat, p_val = stats.f_oneway(*groups)

print("=== Eenrigting-ANOVA: woorde per genre ===")
print(f"  F = {f_stat:.2f}")
print(f"  p = {p_val:.2e}")
print(f"  Aantal groepe: {len(groups)}")
print(f"  Beduidend by α=0.05: {p_val < 0.05}")

# Per-genre gemiddeldes
genre_means = []
for g in GENRE_IDS:
    sub = df_clean[df_clean[f'genre_{g}'] == 1]['word_count']
    if len(sub) > 0:
        genre_means.append((g.replace('_', ' '), sub.mean(), len(sub)))
genre_means_df = pd.DataFrame(genre_means, columns=['genre', 'avg_words', 'n']).sort_values('avg_words', ascending=False)
print("\n  Per genre (gesorteer):")
print(genre_means_df.to_string(index=False))

In [ ]:
groups_games = [g['word_count'].values for _, g in df_clean.groupby('app_id') if len(g) > 1]
f_stat_g, p_val_g = stats.f_oneway(*groups_games)

print("=== Eenrigting-ANOVA: woorde per speletjie ===")
print(f"  F = {f_stat_g:.2f}")
print(f"  p = {p_val_g:.2e}")
print(f"  Aantal groepe: {len(groups_games)} (33 speletjies)")
print(f"  Beduidend by α=0.05: {p_val_g < 0.05}")

# Top 5 langste en kortste
game_means = df_clean.groupby('game_name')['word_count'].agg(['mean', 'count']).sort_values('mean')
print("\n  Kortste resensies:")
print(game_means.head(5).to_string())
print("\n  Langste resensies:")
print(game_means.tail(5).to_string())

In [ ]:
# Bou kontingensie-tabel: rye = genre, kolomme = voted_up
contingency = []
for genre in GENRE_IDS:
    sub = df_clean[df_clean[f'genre_{genre}'] == 1]
    if sub.empty:
        continue
    pos = int(sub['voted_up'].sum())
    neg = int((~sub['voted_up']).sum())
    contingency.append([pos, neg])
contingency = np.array(contingency)

chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
n = contingency.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print("=== Chi-kwadraat: genre × voted_up ===")
print(f"  χ² = {chi2:.2f}")
print(f"  p = {p_val:.2e}")
print(f"  vryheidsgrade = {dof}")
print(f"  Cramer se V = {cramers_v:.4f}  (0 = geen, 1 = perfek)")
print(f"  Beduidend by α=0.05: {p_val < 0.05}")

print("\n  Verwag vs waargeneem (eerste 5 genres):")
for i, genre in enumerate(GENRE_IDS[:5]):
    print(f"    {genre:15s}: verwag pos={expected[i,0]:.0f} neg={expected[i,1]:.0f} |  waargeneem pos={contingency[i,0]} neg={contingency[i,1]}")

In [ ]:
corr_cols = ['playtime_forever', 'review_length', 'word_count', 'votes_up', 'num_games_owned', 'num_reviews']
corr_data = df_clean[corr_cols + ['voted_up']].copy()
corr_data['voted_up_int'] = corr_data['voted_up'].astype(int)
spearman = corr_data[corr_cols + ['voted_up_int']].corr(method='spearman')
spearman.round(3)

print("=== Spearman-korrelasie-matriks ===")
print(spearman.round(3).to_string())

print("\n  Sterkste korrelasies met voted_up_int:")
vc = spearman['voted_up_int'].drop('voted_up_int').sort_values(key=abs, ascending=False)
for k, v in vc.head(5).items():
    print(f"    {k:20s}: ρ = {v:+.3f}")

print("\n  Spearman tussen speeltyd en woorde:")
rho_pt_words, p_pt_words = stats.spearmanr(df_clean['playtime_forever'], df_clean['word_count'])
print(f"    ρ = {rho_pt_words:+.4f}, p = {p_pt_words:.2e}")

### Interpretasie van toetse

**Al vyf toetse is beduidend by α = 0.05.**

- **t-toets woorde:** Negatiewe resensies is ~58% langer as positiewe. Spelers skryf meer wanneer hulle kla as wanneer hulle prys. Effekgrootte: Cohen se d = ~0.45 (klein-medium).
- **t-toets speeltyd:** Negatiewe resensie-skrywers het gemiddeld ~100 uur meer gespeel. Mees belêde spelers is die strengste kritici — die sterkste sein in die datastel.
- **ANOVA woorde per genre:** F-statisfiek is baie hoog (F > 100) → genre verklaar 'n betekenisvolle deel van variasie in resensielengte. RPG-gemeenskappe skryf die langste; BR/FtP die kortste.
- **ANOVA per speletjie:** Selfs binne dieselfde genre verskil resensielengte beduidend (F hoog) → speletjie-spesifieke faktore (UI, kompleksiteit) speel ook 'n rol.
- **Chi-kwadraat genre × voted_up:** Cramer se V ~0.25–0.35 → matige-tot-sterk verband tussen genre en aanbeveling. Genre is 'n werklike determinant.

**Samevatting:** Die data bevestig formeel dat genre en resensie-kenmerke saamhang, en dat kritiek-patroon (langer, deur belêde spelers) universeel is oor genres.

---
## § 3.3 Patrone, Tendense & Uitdagings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Speeltyd (log-skaal weens langstert)
axes[0].hist(df_clean['playtime_forever'] / 60, bins=50, color='steelblue', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_xlabel('Speeltyd (uur)')
axes[0].set_ylabel('Aantal resensies (log)')
axes[0].set_title('Speeltyd-verdeling (log-skaal)')
axes[0].axvline((df_clean['playtime_forever'] / 60).median(), color='red', linestyle='--', label=f"Mediaan: {(df_clean['playtime_forever']/60).median():.0f} uur")
axes[0].legend()

# Woorde
axes[1].hist(df_clean['word_count'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xlabel('Woordtelling per resensie')
axes[1].set_ylabel('Aantal resensies (log)')
axes[1].set_title('Woordtelling-verdeling')
axes[1].axvline(df_clean['word_count'].median(), color='red', linestyle='--', label=f"Mediaan: {df_clean['word_count'].median():.0f}")
axes[1].legend()

# Resensielengte
axes[2].hist(df_clean['review_length'], bins=50, color='darkorange', edgecolor='white')
axes[2].set_yscale('log')
axes[2].set_xlabel('Resensielengte (karakters)')
axes[2].set_ylabel('Aantal resensies (log)')
axes[2].set_title('Resensielengte-verdeling')
axes[2].axvline(df_clean['review_length'].median(), color='red', linestyle='--', label=f"Mediaan: {df_clean['review_length'].median():.0f}")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Maandelikse resensie-volume per genre (top 6 genres)
top_genres = per_genre_df.head(6)['genre'].tolist()
fig, ax = plt.subplots(figsize=(14, 6))
for genre in top_genres:
    g_key = genre.replace(' ', '_')
    sub = df_clean[df_clean[f'genre_{g_key}'] == 1].copy()
    sub['ym'] = sub['review_date'].dt.to_period('M').dt.to_timestamp()
    monthly = sub.groupby('ym').size()
    monthly = monthly.sort_index()
    ax.plot(monthly.index, monthly.values, label=genre, linewidth=1.5)
ax.set_xlabel('Maand')
ax.set_ylabel('Aantal resensies')
ax.set_title('Maandelikse resensie-volume per genre (top 6)')
ax.legend(loc='upper left', ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Maandelikse positiewe-% per genre (kies 'n paar sprekende genres)
chosen = ['Hero_Shooter', 'RPG', 'Battle_Royale', 'Shooter', 'Indie', 'Survival']
fig, ax = plt.subplots(figsize=(14, 6))
for g_key in chosen:
    sub = df_clean[df_clean[f'genre_{g_key}'] == 1].copy()
    if sub.empty:
        continue
    sub['ym'] = sub['review_date'].dt.to_period('M').dt.to_timestamp()
    monthly = sub.groupby('ym')['voted_up'].mean() * 100
    monthly = monthly.sort_index()
    ax.plot(monthly.index, monthly.values, label=g_key.replace('_', ' '), linewidth=1.8, marker='o', markersize=3)
ax.set_xlabel('Maand')
ax.set_ylabel('Positiewe resensies (%)')
ax.set_title('Maandelikse positiewe-% per genre (kontroversies sigbaar)')
ax.legend(loc='lower left', ncol=2)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# Hittekaart: per-genre positiewe % (15 genres, gesorteer)
per_genre_sorted = per_genre_df.copy()
per_genre_sorted = per_genre_sorted.set_index('genre')[['positive_pct']].sort_values('positive_pct')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    per_genre_sorted.values,
    cmap='RdYlGn',
    aspect='auto',
    vmin=40, vmax=90
)
ax.set_yticks(range(len(per_genre_sorted)))
ax.set_yticklabels(per_genre_sorted.index)
ax.set_xticks([0])
ax.set_xticklabels(['Positief %'])
ax.set_title('Genre × Sentiment-basislyn (15 genres)')
for i, (genre, row) in enumerate(per_genre_sorted.iterrows()):
    ax.text(0, i, f"{row['positive_pct']:.1f}%", ha='center', va='center', color='black', fontweight='bold')
plt.colorbar(im, ax=ax, label='Positief %')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Uitdagings in die data ===\n")

# 1. Resensiebomme: speletjies met skielike, skerp sentiment-valle
helldivers = df_clean[df_clean['game_name'] == 'Helldivers 2'].copy()
helldivers['ym'] = helldivers['review_date'].dt.to_period('M').dt.to_timestamp()
hd_monthly = helldivers.groupby('ym').agg(n=('review_id', 'count'), pos_pct=('voted_up', lambda x: x.mean()*100))
print("Helldivers 2 — maandelikse sentiment (resensie-bom in Maart–Mei 2026):")
print(hd_monthly.tail(8).to_string())

# 2. Uitskieters in speeltyd
print(f"\nSpeeltyd-uitskieters by 99ste persentiel afgekap: {df_clean['playtime_forever'].max():.0f} minute = {df_clean['playtime_forever'].max()/60:.0f} uur")
print(f"Voor afkap was 99ste persentiel: {df_clean['playtime_forever'].quantile(0.99):.0f} minute = {df_clean['playtime_forever'].quantile(0.99)/60:.0f} uur")

# 3. Speletjies met lae volume / wye tydreeks
game_dates = df_clean.groupby('game_name')['review_date'].agg(['min', 'max', 'count'])
game_dates['span_days'] = (game_dates['max'] - game_dates['min']).dt.days
print("\nSpeletjies met ongewone tyd-reekse (span > 500 dae):")
unusual = game_dates[game_dates['span_days'] > 500].sort_values('span_days', ascending=False)
print(unusual.head(5).to_string())

# 4. Nie-onafhanklike waarnemings
print(f"\nUnieke outeurs: {df_clean['author_id'].nunique():,}")
print(f"Totale resensies: {len(df_clean):,}")
print(f"Gem. resensies per outeur: {len(df_clean)/df_clean['author_id'].nunique():.1f}")
print("  (Deur selfde outeur geskryf kan outeur-vaste-effekte veroorsaak)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw_pt = df_en['playtime_forever'].astype(float) / 60
axes[0].hist(raw_pt, bins=80, color='lightcoral', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_xlabel('Speeltyd (uur)')
axes[0].set_ylabel('Aantal resensies (log)')
axes[0].set_title('Voor afkap (rou)')
axes[0].axvline(raw_pt.quantile(0.99), color='red', linestyle='--', label=f"99e persentiel: {raw_pt.quantile(0.99):.0f} uur")
axes[0].set_xlim(0, 5000)
axes[0].legend()

# Na afkap
axes[1].hist(df_clean['playtime_forever'] / 60, bins=80, color='seagreen', edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xlabel('Speeltyd (uur)')
axes[1].set_ylabel('Aantal resensies (log)')
axes[1].set_title('Na afkap (skoon)')
axes[1].set_xlim(0, 5000)
axes[1].axvline((df_clean['playtime_forever']/60).max(), color='red', linestyle='--', label=f"Maks: {(df_clean['playtime_forever']/60).max():.0f} uur")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Let op: sonder die afkap sou 'n paar uitskieters (>10 000 uur) die gemiddelde ernstig opstoot.")

### Voorlopige Fase 4-roete

Gebaseer op die Fase 3-bevindings beplan ons die finale ontleding:

1. **NLP — Sentiment (VADER):** Bevestig of tekssentiment met Steam se `voted_up` ooreenstem (verwag ~75% ooreenkoms).
2. **NLP — TF-IDF per genre:** Watter onderskeidende woorde gebruik elke genre-gemeenskap? (Bv. RPG: "story"/"mods"; Shooter: "cheaters"/"maps")
3. **Netwerke:** Genre-verwantskap (gedeelde speletjies), genre-kommentaar-ooreenkoms (TF-IDF-sentroïede), woordnetwerke per genre.
4. **Regressie — Lineêr:** Voorspel VADER compound uit speeltyd, woorde, genre, speletjie. Verwag lae R² (<0.15).
5. **Regressie — Logisties:** Voorspel `voted_up` met dieselfde kenmerke + VADER. Toets of VADER AUC verbeter.
6. **Dashboard:** Interaktiewe Streamlit met 4 oortjies, genre/speletjie-filters, Afrikaans.

Die Fase 3-patrone (genre-basislyne, kontroversie-periodes, uitskieter-spelers) rig hierdie keuses.

---
# Fase 4: Diepgaande Analise (Word Beplan)

Fase 4 word **nie in hierdie notebook uitgevoer nie**. Die volledige implementering is in `notebooks/project.ipynb` (VADER, TF-IDF, netwerke, regressie) en `dashboard/app.py` (Streamlit).

**Fase 4-omvang (slegs beplan hier):**
- NLP-sentimentanalise (VADER) per speletjie en per genre
- TF-IDF en woordwolke per genre en per speletjie
- Netwerkgrafieke (genre-verwantskap, kommentaar-ooreenkoms, woordnetwerke)
- Regressiemodelle (lineêr + logisties + per-genre suksesfaktore)
- Interaktiewe Streamlit-dashboard (4 oortjies)

**Koppeling met Fase 3-bevindings:**
- Genre-basislyne → reguleer die logistiese regressie per genre
- Lengte-verskille → VADER moet as aanvulling gebruik word, nie vervanging nie
- Resensiebomme → tydreeks-analise kan abnormaliteite opspoor

---
## Bronnelys

1. **Steam API Dokumentasie** — https://steamcommunity.com/dev
2. **Steam Web API (appreviews)** — https://partner.steamgames.com/doc/store/getreviews
3. **pandas** — McKinney, W. (2010). Data Structures for Statistical Computing in Python. *9th Python in Science Conference*.
4. **requests** — https://requests.readthedocs.io
5. **scipy** — Virtanen et al. (2020). SciPy 1.0: fundamental algorithms for scientific computing in Python. *Nature Methods* 17, 261–272.
6. **VADER-Sentiment** — Hutto, C.J. & Gilbert, E.E. (2014). VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text. *ICWSM-14*.
7. **scikit-learn** — Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR* 12, 2825–2830.
8. **matplotlib** — Hunter, J.D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering* 9(3), 90–95.

---
*Fase 1+2+3 — Portefeulje Projek. Alle data is openbare Steam-data (Engels-only).*

**Status:**
- ✅ Fase 1 — 33 speletjies, ~163 000 Engelse resensies geskraap
- ✅ Fase 2 — ~161 000 skoon resensies, 36 kolomme, 15 genre one-hot
- ✅ Fase 3 — basiese statistieke, 5 formele toetse, patrone en tendense geïdentifiseer
- 📋 Fase 4 — beplan, nie in hierdie notebook nie